##TODO: Process videos from google cloud
* Apply mediapipe blink detection to each frame per video
* Find high-quality non-blink frame to represent the video, rather than center image. Upload that for JPG component.
* Alongside, upload parquet of blink/not blink. Classify as blink if there are no other non-blink frames in the video.

## Get data from Google Cloud

In [4]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
#!mkdir -p permissions
#!gcloud storage cp "gs://webcam-jspsych-frames/v1/include_list.csv" permissions/

Copying gs://webcam-jspsych-frames/v1/include_list.csv to file://permissions/include_list.csv


To take a quick anonymous survey, run:
  $ gcloud survey



In [ ]:
#!mkdir -p eyedata23
#!gcloud storage cp "gs://webcam-jspsych-webm/eyedata23/*.tar" eyedata23/

Copying gs://webcam-jspsych-webm/eyedata23/01mfzkag.tar to file://eyedata23/01mfzkag.tar
Copying gs://webcam-jspsych-webm/eyedata23/032aobu5.tar to file://eyedata23/032aobu5.tar
Copying gs://webcam-jspsych-webm/eyedata23/03f6hxag.tar to file://eyedata23/03f6hxag.tar
Copying gs://webcam-jspsych-webm/eyedata23/043z3m7m.tar to file://eyedata23/043z3m7m.tar
Copying gs://webcam-jspsych-webm/eyedata23/04hntjve.tar to file://eyedata23/04hntjve.tar
Copying gs://webcam-jspsych-webm/eyedata23/05yp8162.tar to file://eyedata23/05yp8162.tar
Copying gs://webcam-jspsych-webm/eyedata23/06zbm2q6.tar to file://eyedata23/06zbm2q6.tar
Copying gs://webcam-jspsych-webm/eyedata23/07ulcjuu.tar to file://eyedata23/07ulcjuu.tar
Copying gs://webcam-jspsych-webm/eyedata23/08bz6mem.tar to file://eyedata23/08bz6mem.tar
Copying gs://webcam-jspsych-webm/eyedata23/08jbvotj.tar to file://eyedata23/08jbvotj.tar
Copying gs://webcam-jspsych-webm/eyedata23/08va3b9z.tar to file://eyedata23/08va3b9z.tar
Copying gs://webcam-j

In [ ]:
#import glob, subprocess
#for tar in glob.glob("eyedata23/*.tar"):
    #subprocess.run(["tar", "xf", tar, "-C", "eyedata23"])
#print("done — extracted", len(glob.glob("eyedata23/*.webm")), "video clips")

done — extracted 296579 video clips


In [ ]:
# !mkdir -p eyedata24
# !gcloud storage cp "gs://webcam-jspsych-webm/eyedata24/*.tar" eyedata24/

Streaming output truncated to the last 5000 lines.
Copying gs://webcam-jspsych-webm/eyedata24/ak2sqkl9.tar to file://eyedata24/ak2sqkl9.tar
Copying gs://webcam-jspsych-webm/eyedata24/al6acv1s.tar to file://eyedata24/al6acv1s.tar
Copying gs://webcam-jspsych-webm/eyedata24/aqdp5bje.tar to file://eyedata24/aqdp5bje.tar
Copying gs://webcam-jspsych-webm/eyedata24/awq75ff2.tar to file://eyedata24/awq75ff2.tar
Copying gs://webcam-jspsych-webm/eyedata24/aywb3d8b.tar to file://eyedata24/aywb3d8b.tar
Copying gs://webcam-jspsych-webm/eyedata24/b0k95bso.tar to file://eyedata24/b0k95bso.tar
Copying gs://webcam-jspsych-webm/eyedata24/b5lvb21n.tar to file://eyedata24/b5lvb21n.tar
Copying gs://webcam-jspsych-webm/eyedata24/b6jr7puj.tar to file://eyedata24/b6jr7puj.tar
Copying gs://webcam-jspsych-webm/eyedata24/bbe1kuel.tar to file://eyedata24/bbe1kuel.tar
Copying gs://webcam-jspsych-webm/eyedata24/bju3jtuf.tar to file://eyedata24/bju3jtuf.tar
Copying gs://webcam-jspsych-webm/eyedata24/bqruh864.tar to 

In [ ]:
# import glob, subprocess
# for tar in glob.glob("eyedata24/*.tar"):
#     subprocess.run(["tar", "xf", tar, "-C", "eyedata24"])
# print("done — extracted", len(glob.glob("eyedata24/*.webm")), "video clips")

done — extracted 95909 video clips


##Process video clips

In [10]:
%pip install osfclient
%pip install mediapipe
%pip install fastparquet

In [ ]:
# Replace the placeholder URL with the actual URL of the eyedata repository
!git clone https://github.com/jspsych/eyedata.git

Cloning into 'eyedata'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 363 (delta 24), reused 30 (delta 17), pack-reused 313 (from 1)
Receiving objects: 100% (363/363), 300.26 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (154/154), done.


In [14]:
import os
import re
import tarfile
from pathlib import Path
import pandas as pd
from osfclient.api import OSF

import importlib.util
import sys
import cv2
import numpy as np

import glob
import math
import shutil
import subprocess

In [8]:
# Define the absolute path to the script
# The script was not found at the previous location. Based on tar extraction, it's likely under 'eyedata24/'.
script_path = 'eyedata/data-processing/blink_utils.py'

# Load the module dynamically
spec = importlib.util.spec_from_file_location("blink_utils", script_path)
blink_utils = importlib.util.module_from_spec(spec)
sys.modules["blink_utils"] = blink_utils
spec.loader.exec_module(blink_utils)

FileNotFoundError: [Errno 2] No such file or directory: '/content/eyedata/data-processing/blink_utils.py'

In [ ]:
# Get model
!wget -O face_landmarker.task -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

In [ ]:
def laplace_var(frame):
    """
    Returns laplacian variance of the frame.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance

def mean_brightness(frame):
    """
    Return mean frame brightness.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean_brightness = np.mean(gray)
    return mean_brightness

In [17]:
# --- Configurations ---
OSF_TOKEN = "xHQmxKU1s9ocTj4hiu0OozjpWQeM0Wdlnkq64GG6ruqjk230hR4rnQmcOV1NxJHdqksuYz"
OSF_NODES = {
    'public': {'train': 'apm3t', 'test': 'vay5f'},
    'private': {'train': 'syrvb', 'test': '6dtnm'}
}

BASE_DRIVE_DIR = Path('/content/drive/MyDrive/Eye_tracking_work/Data_processing')

# --- 2. Permissions & Subject Designation ---
#perm_df = pd.read_csv('permissions/include_list.csv')
# public_subjects = {
#     'train': set(perm_df[perm_df['dataset'] == 'eyedata23']['subject_id']),
#     'test': set(perm_df[perm_df['dataset'] == 'eyedata24']['subject_id'])
# }

# --- Heuristics for Decent Frame ---
def laplace_var(frame):
    return cv2.Laplacian(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

def mean_brightness(frame):
    return np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))

def is_decent(frame, blink_score):
    if frame is None or blink_score is None: return False
    if blink_score > 0.3: return False  # Permissive blink threshold
    laplace = laplace_var(frame)
    bright = mean_brightness(frame)
    if laplace < 10.0: return False     # Not entirely blurry
    if bright < 5.0 or bright > 300.0: return False # Not entirely black/white
    return True

In [ ]:
def find_decent_frame(video_path, blink_scores):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Check for corrupted metadata (e.g. frame counts in the millions/billions)
    # Most of these clips are < 10 seconds. Anything > 100,000 is likely a metadata error.
    is_metadata_corrupt = total_frames <= 0 or total_frames > 100000

    def check_frame(idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: return None, -1, -1
        b_score = blink_scores[idx] if 0 <= idx < len(blink_scores) else 0.0
        return frame, b_score, (laplace_var(frame) * (1 - b_score))

    best_frame, best_score, best_comp = None, None, -1

    if is_metadata_corrupt:
        # Fallback: Just try the first 15 frames sequentially
        for i in range(15):
            ret, frame = cap.read()
            if not ret: break
            b_score = blink_scores[i] if i < len(blink_scores) else 0.0
            if is_decent(frame, b_score):
                cap.release()
                return frame, b_score

            comp = (laplace_var(frame) * (1 - b_score))
            if comp > best_comp:
                best_comp = comp
                best_frame = frame
                best_score = b_score
    else:
        # Standard Binary search heuristic: center, then quarters
        search_points = [total_frames // 2, total_frames // 4, 3 * total_frames // 4]
        for pt in search_points:
            frame, b_score, comp = check_frame(pt)
            if frame is not None:
                if is_decent(frame, b_score):
                    cap.release()
                    return frame, b_score
                if comp > best_comp:
                    best_comp = comp
                    best_frame = frame
                    best_score = b_score

    cap.release()

    # Usability fallback
    if best_frame is not None:
        bright = mean_brightness(best_frame)
        if 2.0 < bright < 253.0:
            return best_frame, best_score

    return None, None

In [ ]:
# --- Pipeline Execution ---
def run_pipeline(dataset_dir, split_type):
    print(f"\nStarting pipeline for {split_type} ({dataset_dir})")
    all_files = sorted(glob.glob(f"{dataset_dir}/*.webm"))

    public_dir = BASE_DRIVE_DIR / 'public' / split_type
    private_dir = BASE_DRIVE_DIR / 'private' / split_type
    public_dir.mkdir(parents=True, exist_ok=True)
    private_dir.mkdir(parents=True, exist_ok=True)

    metadata = {'public': [], 'private': []}
    processed_subjects = set()

    # Load existing metadata to resume gracefully
    for access in ['public', 'private']:
        meta_path_parquet = BASE_DRIVE_DIR / access / split_type / f"metadata_{split_type}.parquet"
        meta_path_csv = BASE_DRIVE_DIR / access / split_type / f"metadata_{split_type}.csv"

        if meta_path_parquet.exists():
            try:
                existing_df = pd.read_parquet(meta_path_parquet)
                metadata[access] = existing_df.to_dict('records')
                processed_subjects.update(existing_df['subject_id'].unique())
                print(f"Loaded {len(existing_df)} existing metadata records (parquet) for {access} {split_type}.")
            except Exception as e:
                print(f"Could not load existing parquet metadata for {access}: {e}")
        elif meta_path_csv.exists():
            try:
                existing_df = pd.read_csv(meta_path_csv)
                metadata[access] = existing_df.to_dict('records')
                processed_subjects.update(existing_df['subject_id'].unique())
                print(f"Loaded {len(existing_df)} existing metadata records (csv) for {access} {split_type}.")
            except Exception as e:
                print(f"Could not load existing csv metadata for {access}: {e}")

    # Group files by subject
    from collections import defaultdict
    import tempfile

    subject_files = defaultdict(list)
    for f in all_files:
        fname = os.path.basename(f)
        subj = fname.split('_')[0]
        subject_files[subj].append(f)

    print(f"Found {len(subject_files)} subjects. {len(processed_subjects)} already processed based on metadata.")

    for i, (subject_id, files) in enumerate(subject_files.items()):
        if subject_id in processed_subjects:
            continue

        is_pub = subject_id in public_subjects[split_type]
        access = 'public' if is_pub else 'private'
        target_dir = public_dir if is_pub else private_dir

        # Additional check in case metadata wasn't saved but tar exists
        tar_name = f"{subject_id}_jpg.tar.gz"
        tar_path = target_dir / tar_name
        if tar_path.exists():
            print(f"Subject {subject_id} tar already exists, skipping ({i+1}/{len(subject_files)})")
            continue

        print(f"Processing subject {subject_id} ({i+1}/{len(subject_files)})")

        subject_metadata = []
        subject_frames = {}
        skip_subject = False

        for file_path in files:
            fname = os.path.basename(file_path)
            # Process frame
            blink_scores = blink_utils.extract_blendshape_blink_seq(file_path) if 'blink_utils' in globals() else [0]*100
            frame, b_score = find_decent_frame(file_path, blink_scores)

            if frame is None:
                print(f"No decent frame found for {fname}. Skipping subject {subject_id}.")
                skip_subject = True
                break

            subject_frames[fname] = frame
            subject_metadata.append({'subject_id': subject_id, 'file': fname, 'blink_score': b_score})

        if skip_subject:
            continue

        # Save to Drive as a subject-specific chunk
        with tempfile.TemporaryDirectory() as tmpdirname:
            tmp_dir = Path(tmpdirname)
            subj_jpgs = []
            for fname, frame in subject_frames.items():
                out_name = f"{fname.split('.')[0]}.jpg"
                cv2.imwrite(str(tmp_dir / out_name), frame)
                subj_jpgs.append(out_name)

            subprocess.run(["tar", "-czf", str(tar_path), "-C", str(tmp_dir)] + subj_jpgs)

        metadata[access].extend(subject_metadata)

        # Optional: Save metadata incrementally as parquet
        if (i + 1) % 50 == 0 or (i + 1) == len(subject_files):
            pd.DataFrame(metadata[access]).to_parquet(target_dir / f"metadata_{split_type}.parquet", index=False)

    # Final metadata save
    for access in ['public', 'private']:
        if metadata[access]:
            pd.DataFrame(metadata[access]).to_parquet(BASE_DRIVE_DIR / access / split_type / f"metadata_{split_type}.parquet", index=False)

    # --- 4. Upload to OSF ---
    osf = OSF(token=OSF_TOKEN)
    for access in ['public', 'private']:
        if not metadata[access]: continue

        node_id = OSF_NODES[access][split_type]
        print(f"Uploading {access} {split_type} data to OSF node {node_id}...")
        try:
            project = osf.project(node_id)
            storage = project.storage('osfstorage')

            def upload_or_update(local_path, filename):
                existing_files = {f.name: f for f in storage.files}
                if filename in existing_files:
                    print(f"Updating existing {filename}...")
                    with open(local_path, 'rb') as f:
                        existing_files[filename].update(f)
                else:
                    print(f"Creating new {filename}...")
                    with open(local_path, 'rb') as f:
                        storage.create_file(filename, f)

            # Save and upload metadata parquet
            meta_path = BASE_DRIVE_DIR / access / split_type / f"metadata_{split_type}.parquet"
            if meta_path.exists():
                upload_or_update(meta_path, f"metadata_{split_type}.parquet")

            # Upload tar chunks per subject directly from Drive
            chunk_dir = BASE_DRIVE_DIR / access / split_type
            df = pd.DataFrame(metadata[access])
            subjects = df['subject_id'].unique()

            for subj in subjects:
                actual_tar_name = f"{subj}_jpg.tar.gz"
                tar_path = chunk_dir / actual_tar_name

                if tar_path.exists():
                    upload_or_update(tar_path, actual_tar_name)

            print(f"Successfully uploaded {access} {split_type}!")
        except Exception as e:
            print(f"Failed OSF upload for {access} {split_type}: {e}")


In [ ]:
# Run pipeline
run_pipeline('eyedata23', 'train')
#run_pipeline('eyedata24', 'test')


Starting pipeline for test (eyedata24)
Loaded 36000 existing metadata records (csv) for public test.
Loaded 10599 existing metadata records (csv) for private test.
Found 352 subjects. 171 already processed based on metadata.
Processing subject a0pvejzf (84/352)
No decent frame found for a0pvejzf_1_11_16.25.webm. Skipping subject a0pvejzf.
Processing subject flbsoj0l (144/352)
No decent frame found for flbsoj0l_1_11_16.25.webm. Skipping subject flbsoj0l.
Processing subject g61ntexm (151/352)
No decent frame found for g61ntexm_1_11_16.25.webm. Skipping subject g61ntexm.
Processing subject hncaqa8m (163/352)
No decent frame found for hncaqa8m_1_11_16.25.webm. Skipping subject hncaqa8m.
Processing subject k28vh0vm (176/352)
Processing subject k5mc27ug (177/352)
Processing subject k5mk3y3e (178/352)
Processing subject k5v9b5a9 (179/352)
Processing subject k9x2l6b6 (180/352)
Processing subject kb4328pl (181/352)
Processing subject kc8pxvsz (182/352)
Processing subject kpmmbc8y (183/352)
Pro

In [20]:
from osfclient.api import OSF
from pathlib import Path
import traceback

def sync_drive_tars_to_osf(base_drive_dir, osf_nodes, osf_token):
    """
    Checks if subject tars from Google Drive exist in their respective OSF components.
    Uploads them if they are missing.
    """
    osf = OSF(token=osf_token)

    for access in ['public', 'private']:
        for split_type in ['train', 'test']:
            node_id = osf_nodes.get(access, {}).get(split_type)
            if not node_id:
                continue

            drive_dir = Path(base_drive_dir) / access / split_type
            if not drive_dir.exists():
                print(f"Directory {drive_dir} does not exist. Skipping {access} {split_type}.")
                continue

            print(f"\nChecking {access} {split_type} (OSF Node: {node_id})...")
            try:
                project = osf.project(node_id)
                storage = project.storage('osfstorage')

                # Retrieve list of files already in OSF
                existing_files = {f.name for f in storage.files}

                # Get all tar.gz files in the drive directory
                tar_files = list(drive_dir.glob("*_jpg.tar.gz"))
                print(f"Found {len(tar_files)} tar files locally.")

                uploaded_count = 0
                for tar_path in tar_files:
                    filename = tar_path.name
                    if filename not in existing_files:
                        print(f"  Uploading missing file: {filename}...")
                        try:
                            with open(tar_path, 'rb') as f:
                                storage.create_file(filename, f)
                            uploaded_count += 1
                        except Exception as file_e:
                            print(f"    -> Failed to upload {filename}: {file_e}")
                    else:
                        # print(f"  File {filename} already exists. Skipping.")
                        pass

                # Also check metadata files just in case
                for meta_ext in ['parquet', 'csv']:
                    meta_file = f"metadata_{split_type}.{meta_ext}"
                    meta_path = drive_dir / meta_file
                    if meta_path.exists() and meta_file not in existing_files:
                        print(f"  Uploading missing metadata: {meta_file}...")
                        try:
                            with open(meta_path, 'rb') as f:
                                storage.create_file(meta_file, f)
                        except Exception as meta_e:
                            print(f"    -> Failed to upload {meta_file}: {meta_e}")

                print(f"Finished {access} {split_type}: Uploaded {uploaded_count} new tar files.")

            except Exception as node_e:
                print(f"Failed to process OSF node {node_id} for {access} {split_type}: {node_e}")
                traceback.print_exc()

# Run the sync function using the previously defined configuration variables
# sync_drive_tars_to_osf(BASE_DRIVE_DIR, OSF_NODES, OSF_TOKEN)

sync_drive_tars_to_osf(BASE_DRIVE_DIR, OSF_NODES, OSF_TOKEN)



Checking public train (OSF Node: apm3t)...
Found 1745 tar files locally.
  Uploading missing file: fh14tc1r_jpg.tar.gz...
    -> Failed to upload fh14tc1r_jpg.tar.gz: fh14tc1r_jpg.tar.gz
  Uploading missing file: fp5xdlqf_jpg.tar.gz...
    -> Failed to upload fp5xdlqf_jpg.tar.gz: fp5xdlqf_jpg.tar.gz
  Uploading missing file: fvt730y0_jpg.tar.gz...
    -> Failed to upload fvt730y0_jpg.tar.gz: fvt730y0_jpg.tar.gz
  Uploading missing file: g1buv670_jpg.tar.gz...
    -> Failed to upload g1buv670_jpg.tar.gz: g1buv670_jpg.tar.gz
  Uploading missing file: g66gl7mv_jpg.tar.gz...
    -> Failed to upload g66gl7mv_jpg.tar.gz: g66gl7mv_jpg.tar.gz
  Uploading missing file: g9ekv1vx_jpg.tar.gz...
    -> Failed to upload g9ekv1vx_jpg.tar.gz: g9ekv1vx_jpg.tar.gz
  Uploading missing file: ghb88gph_jpg.tar.gz...
    -> Failed to upload ghb88gph_jpg.tar.gz: ghb88gph_jpg.tar.gz
  Uploading missing file: go1g73kn_jpg.tar.gz...
    -> Failed to upload go1g73kn_jpg.tar.gz: go1g73kn_jpg.tar.gz
  Uploading mi

In [21]:
from osfclient.api import OSF
from pathlib import Path
import traceback

# Investigate one of the failing files
test_filename = 'fh14tc1r_jpg.tar.gz'
drive_dir = Path(BASE_DRIVE_DIR) / 'public' / 'train'
test_filepath = drive_dir / test_filename

print(f"Investigating {test_filename}...")
if test_filepath.exists():
    print(f"File exists locally.")
    print(f"Size: {test_filepath.stat().st_size} bytes")

    print("\nAttempting manual upload to OSF Node 'apm3t'...")
    try:
        osf = OSF(token=OSF_TOKEN)
        project = osf.project('apm3t')
        storage = project.storage('osfstorage')

        with open(test_filepath, 'rb') as f:
            storage.create_file(test_filename, f)
        print("Upload successful!")
    except Exception as e:
        print(f"\nUpload failed.")
        print(f"Exception Type: {type(e).__name__}")
        print("Full traceback:\n")
        traceback.print_exc()
else:
    print(f"File {test_filepath} not found locally.")


Investigating fh14tc1r_jpg.tar.gz...
File exists locally.
Size: 11626641 bytes

Attempting manual upload to OSF Node 'apm3t'...

Upload failed.
Exception Type: FileExistsError
Full traceback:



Traceback (most recent call last):
  File "/tmp/ipykernel_1534/4290967016.py", line 22, in <cell line: 0>
    storage.create_file(test_filename, f)
  File "/usr/local/lib/python3.12/dist-packages/osfclient/models/storage.py", line 113, in create_file
    raise FileExistsError(path)
FileExistsError: fh14tc1r_jpg.tar.gz


In [22]:
import os
from pathlib import Path

def get_dir_size(path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

base_path = Path(BASE_DRIVE_DIR)

print("Local Directory Sizes:")
for access in ['public', 'private']:
    for split in ['train', 'test']:
        dir_path = base_path / access / split
        if dir_path.exists():
            size_bytes = get_dir_size(dir_path)
            size_gb = size_bytes / (1024 ** 3)
            print(f"  {access}/{split}: {size_gb:.2f} GB ({size_bytes} bytes)")
        else:
            print(f"  {access}/{split}: Directory does not exist.")


Local Directory Sizes:
  public/train: 16.42 GB (17632295616 bytes)
  public/test: 4.92 GB (5281162385 bytes)
  private/train: 2.92 GB (3136155009 bytes)
  private/test: 1.48 GB (1592304448 bytes)


## Private component hit limit. Need to process public/private data from drive, and extract only high-quality private data to a separate component.

In [25]:
import pandas as pd
from pathlib import Path
from osfclient.api import OSF

osf = OSF(token=OSF_TOKEN)
base_path = Path(BASE_DRIVE_DIR)

targets = {
    'public': 'cbmpq',
    'private': 'w3p2v'
}

for access, node_id in targets.items():
    print(f"\nProcessing {access} metadata...")
    dfs = []
    for split in ['train', 'test']:
        meta_path = base_path / access / split / f"metadata_{split}.parquet"
        if meta_path.exists():
            print(f"  Loaded {split} metadata.")
            df = pd.read_parquet(meta_path)
            df['split'] = split
            dfs.append(df)
        else:
            print(f"  {split} metadata not found at {meta_path}.")

    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        combined_filename = f"combined_metadata_{access}.parquet"
        combined_path = base_path / access / combined_filename

        # Save combined dataframe locally/to Drive
        combined_df.to_parquet(combined_path, index=False)
        print(f"  Combined metadata saved locally. Total rows: {len(combined_df)}")

        print(f"  Uploading to OSF Node '{node_id}'...")
        try:
            project = osf.project(node_id)
            storage = project.storage('osfstorage')

            existing_files = {f.name: f for f in storage.files}
            with open(combined_path, 'rb') as f:
                if combined_filename in existing_files:
                    print(f"  Updating existing {combined_filename} on OSF...")
                    existing_files[combined_filename].update(f)
                else:
                    print(f"  Creating new {combined_filename} on OSF...")
                    storage.create_file(combined_filename, f)
            print(f"  Successfully uploaded {combined_filename} to {node_id}!")
        except Exception as e:
            print(f"  Upload failed: {e}")
    else:
        print(f"  No metadata found for {access}. Skipping upload.")



Processing public metadata...
  train metadata not found at /content/drive/MyDrive/Eye_tracking_work/Data_processing/public/train/metadata_train.parquet.
  Loaded test metadata.
  Combined metadata saved locally. Total rows: 72864
  Uploading to OSF Node 'cbmpq'...
  Creating new combined_metadata_public.parquet on OSF...
  Successfully uploaded combined_metadata_public.parquet to cbmpq!

Processing private metadata...
  train metadata not found at /content/drive/MyDrive/Eye_tracking_work/Data_processing/private/train/metadata_train.parquet.
  Loaded test metadata.
  Combined metadata saved locally. Total rows: 21637
  Uploading to OSF Node 'w3p2v'...
  Creating new combined_metadata_private.parquet on OSF...
  Successfully uploaded combined_metadata_private.parquet to w3p2v!


In [26]:
import pandas as pd
from pathlib import Path

base_path = Path(BASE_DRIVE_DIR)
combined_public_path = base_path / 'public' / 'combined_metadata_public.parquet'

if combined_public_path.exists():
    public_combined_df = pd.read_parquet(combined_public_path)
    print("Head of combined_metadata_public.parquet:")
    display(public_combined_df.head())
else:
    print(f"File not found: {combined_public_path}")


Head of combined_metadata_public.parquet:


,subject_id,file,blink_score,split
0,065fta2n,065fta2n_1_11_16.25.webm,0.091078,test
1,065fta2n,065fta2n_1_11_27.5.webm,0.114680,test
2,065fta2n,065fta2n_1_11_38.75.webm,0.110548,test
3,065fta2n,065fta2n_1_11_5.webm,0.109102,test
4,065fta2n,065fta2n_1_11_50.webm,0.164631,test


## Proposed pipeline:
1. Pull data from gs://webcam-jspsych-webm/eyedata23 (train) and gs://webcam-jspsych-webm/eyedata24 (test/validation).
2 Get permissions from gs://webcam-jspsych-frames/v1/include_list.csv
  * Contains all subjects with permissions in "subject_id" column. Dataset (eyedata23/eyedata24) designated in the "dataset" column.
  * Use to designate subjects as public vs. private. Public if they are included in the csv, private otherwise.
3. Process video data.
  * Use thresholds (be pretty permissive. I want to filter out very bad data, but I also want to get pretty decent frame from each video, but don't sacrifice too much time or usable data trying to do that) and binary search on frames to speed up.
  * If center frame is decent quality (according to arbitrary threshold blink score, laplacian variance, and brightness), use it.
  * Else, split the frames and check centers of first and second split. If one is decent, use it, otherwise choose higher-quality one (based on composite score) and split that section.
  * Iterate until you find okay (just usable, not exactly decent, but cannot be junk i.e. just white, just black, entirely blurry, or completely blinking) image. If you don't, skip the video. Mark that subject as having incomplete data, and skip them in future processing.
  * Save the data in chunks on google drive a la cell 11 (2Gb MAX per chunk, saving in smaller chunks intermittently in case of a crash). Save to "/content/drive/MyDrive/Eye_tracking_work/Data_processing/{public or private}/{train or test}".
4. Once all data is saved to drive, upload to respective OSF components.
  * Public test component: "vay5f"
  * Public train component: "apm3t"
  * Private test component: "6dtnm"
  * Private train component: "syrvb"